In [1]:
%load_ext autoreload
%autoreload 2

from src.DicomRTHandler import DicomRTHandler
from src.config_presets.tools.load_config import load_config
import numpy as np
import os
from src.preprocessing.main import preprocess_numpy_data
from tqdm import tqdm

config = load_config('HNC/UMCG_data.yaml')

src\config_presets\HNC/UMCG_data.yaml


In [17]:


PRIMA_base_dir = r"\\zkh\appdata\RTDicom\Projectline_HNC_modelling\PRI2MA\Structured_wave2"
PRIMA_destination_dir = r"\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\Z. Other stuff\PRIMA_data_checks\PRIMA_preprocessed_stride_rt"
patient_ids_list = os.listdir(PRIMA_base_dir)
patient_ids_list.sort()
len(patient_ids_list)

289

In [18]:
list_one = ['4628069', '5768025', '7822891', '3608845', '1097433', '0020715', '5896858', '2087361', '2083033', '9577277', '3274619', '0345451', '1872399', '2259310', '2376970', '9016977', '4821215', '4054457']
list_two = ['1178564', '9445856']
list_three = ['2076094', '2080453', '5262401', '7985090', '2120032', '3979255', '2199947', '9692367', '2002365', '0454580', '2358949', '7693337', '2365752', '2393545', '2394651', '8303802', '9115342', '0643911']


total_list = list_one + list_two + list_three
total_list = list(set(total_list))  # remove duplicates


In [19]:
temp_list = ['7822891']

for patient_id in tqdm(total_list):
    if patient_id not in patient_ids_list:
        continue
    else:
        print(f"Processing patient: {patient_id}")

    patient_source_dir = os.path.join(PRIMA_base_dir, patient_id)
    patient_subdirs = os.listdir(patient_source_dir)
    patient_subdirs.sort(reverse=True)  # Sort in reverse order to prioritize the most recent subdirectory

    destination_dir_patient = os.path.join(PRIMA_destination_dir, patient_id)
    os.makedirs(destination_dir_patient, exist_ok=True)

    def find_modality_folder(modality):
        for contrast_folder in ("with_contrast", "no_contrast"):
            folder_path = os.path.join(patient_source_dir, contrast_folder)
            for root, dirs, _ in os.walk(folder_path):
                if modality in dirs:
                    return os.path.join(root, modality), contrast_folder
            #continue
            # If we reach here, it means the modality was not found in the current contrast folder
        return None, None

    ct_folder_dir, ct_source = find_modality_folder("CT")
    rtdose_folder_path, rtdose_source = find_modality_folder("RTDOSE")
    RTSTRUCT_dir, rtstruct_source = find_modality_folder("RTSTRUCT")

    if not all((ct_folder_dir, rtdose_folder_path, RTSTRUCT_dir)):
        print(f"Missing CT, RTDOSE, or RTSTRUCT for patient {patient_id}")
        continue

    # look for the RTDOSE file in the RTDOSE folder, to get its path
    rtdose_file_path = os.path.join(rtdose_folder_path, os.listdir(rtdose_folder_path)[0])
    

    handler = DicomRTHandler(config, verbose=False)

    # # Load data 

    # # load the CT files
    handler.load_ct_series(ct_folder_dir)
    handler.load_rtdose(rtdose_file_path)
    handler.load_rtstructs(RTSTRUCT_dir, DLC_folder_dir = None, fix_uids=True)


    preprocessed_CT, preprocessed_RTSTRUCT, preprocessed_RTDOSE = preprocess_numpy_data(config, handler)

    dest_subdir_patient = os.path.join(PRIMA_destination_dir, patient_id)
    os.makedirs(dest_subdir_patient, exist_ok=True)

    np.save(os.path.join(dest_subdir_patient, f"CT.npy"), preprocessed_CT)
    np.save(os.path.join(dest_subdir_patient, f"RTSTRUCT.npy"), preprocessed_RTSTRUCT)
    np.save(os.path.join(dest_subdir_patient, f"RTDOSE.npy"), preprocessed_RTDOSE)


    del handler, preprocessed_CT, preprocessed_RTSTRUCT, preprocessed_RTDOSE



  0%|          | 0/38 [00:00<?, ?it/s]

Processing patient: 0454580
 -- applying spacing correction --
   Original spacing: (0.9765625, 0.9765625, 2.0)
Resampling image from spacing (0.9765625, 0.9765625, 2.0) to [1, 1, 2]
Resampling image from spacing (0.9765625, 0.9765625, 2.0) to [1, 1, 2]
Resampling image from spacing (0.9765625, 0.9765625, 2.0) to [1, 1, 2]
   Corrected spacing: (1.0, 1.0, 2.0)
OARs in z direction: ['parotis_li', 'parotis_re', 'crico', 'thyroid', 'mandible']
OAR values in z direction: [1, 2, 5, 6, 7]
upper_coord: 128, lower_coord: 39
OARs in x direction: ['parotis_li', 'parotis_re']
OAR values in x direction: [1, 2]
upper_coord: 258, lower_coord: 185
OARs in y direction: ['parotis_li', 'parotis_re', 'submandibularis_li', 'submandibularis_re']
OAR values in y direction: [1, 2, 3, 4]
upper_coord: 322, lower_coord: 174


  8%|▊         | 3/38 [01:16<14:57, 25.64s/it]

Processing patient: 2376970
 -- applying spacing correction --
   Original spacing: (0.9765625, 0.9765625, 2.0)
Resampling image from spacing (0.9765625, 0.9765625, 2.0) to [1, 1, 2]
Resampling image from spacing (0.9765625, 0.9765625, 2.0) to [1, 1, 2]
Resampling image from spacing (0.9765625, 0.9765625, 2.0) to [1, 1, 2]
   Corrected spacing: (1.0, 1.0, 2.0)
OARs in z direction: ['parotis_li', 'parotis_re', 'crico', 'thyroid', 'mandible']
OAR values in z direction: [1, 2, 5, 6, 7]
upper_coord: 135, lower_coord: 39
OARs in x direction: ['parotis_li', 'parotis_re']
OAR values in x direction: [1, 2]
upper_coord: 245, lower_coord: 191
OARs in y direction: ['parotis_li', 'parotis_re', 'submandibularis_li', 'submandibularis_re']
OAR values in y direction: [1, 2, 3, 4]
upper_coord: 331, lower_coord: 184


 11%|█         | 4/38 [02:21<21:37, 38.17s/it]

...

 82%|████████▏ | 31/38 [13:49<04:32, 38.91s/it]

Processing patient: 7693337
 -- applying spacing correction --
   Original spacing: (0.9765625, 0.9765625, 2.0)
Resampling image from spacing (0.9765625, 0.9765625, 2.0) to [1, 1, 2]
Resampling image from spacing (0.9765625, 0.9765625, 2.0) to [1, 1, 2]
Resampling image from spacing (0.9765625, 0.9765625, 2.0) to [1, 1, 2]
   Corrected spacing: (1.0, 1.0, 2.0)
OARs in z direction: ['parotis_li', 'parotis_re', 'crico', 'thyroid', 'mandible']
OAR values in z direction: [1, 2, 5, 6, 7]
upper_coord: 141, lower_coord: 56
OARs in x direction: ['parotis_li', 'parotis_re']
OAR values in x direction: [1, 2]
upper_coord: 257, lower_coord: 201
OARs in y direction: ['parotis_li', 'parotis_re', 'submandibularis_li', 'submandibularis_re']
OAR values in y direction: [1, 2, 3, 4]
upper_coord: 320, lower_coord: 173


 84%|████████▍ | 32/38 [15:21<04:52, 48.74s/it]

Processing patient: 9016977
 -- applying spacing correction --
   Original spacing: (0.9765625, 0.9765625, 2.0)
Resampling image from spacing (0.9765625, 0.9765625, 2.0) to [1, 1, 2]
Resampling image from spacing (0.9765625, 0.9765625, 2.0) to [1, 1, 2]
Resampling image from spacing (0.9765625, 0.9765625, 2.0) to [1, 1, 2]
   Corrected spacing: (1.0, 1.0, 2.0)
OARs in z direction: ['parotis_li', 'parotis_re', 'crico', 'thyroid', 'mandible']
OAR values in z direction: [1, 2, 5, 6, 7]
upper_coord: 115, lower_coord: 31
OARs in x direction: ['parotis_li', 'parotis_re']
OAR values in x direction: [1, 2]
upper_coord: 251, lower_coord: 183
OARs in y direction: ['parotis_li', 'parotis_re', 'submandibularis_li', 'submandibularis_re']
OAR values in y direction: [1, 2, 3, 4]
upper_coord: 312, lower_coord: 182


100%|██████████| 38/38 [16:35<00:00, 26.20s/it]
